In [15]:
# Célula 1 — imports e utilitários comuns

from __future__ import annotations

import random
from collections import defaultdict
from pathlib import Path
from typing import List, Sequence, Tuple

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.transforms import InterpolationMode
%load_ext tensorboard

class CenterCropSquare:
    def __call__(self, img: Image.Image):
        w, h = img.size
        side = min(w, h)
        left = (w - side) // 2
        top = (h - side) // 2
        return img.crop((left, top, left + side, top + side))


def stratified_split_indices_3way(
    targets: Sequence[int],
    val_split: float = 0.1,
    test_split: float = 0.1,
    seed: int = 42,
) -> Tuple[List[int], List[int], List[int]]:
    if val_split + test_split >= 1.0:
        raise ValueError("val_split + test_split precisa ser < 1.")

    rng = random.Random(seed)
    by_class = defaultdict(list)

    for idx, label in enumerate(targets):
        by_class[int(label)].append(idx)

    train_idx, val_idx, test_idx = [], [], []

    for indices in by_class.values():
        rng.shuffle(indices)
        n = len(indices)

        if n == 1:
            train_idx.extend(indices)
            continue

        if n == 2:
            train_idx.append(indices[0])
            val_idx.append(indices[1])
            continue

        n_val = max(1, int(round(n * val_split)))
        n_test = max(1, int(round(n * test_split)))

        if n_val + n_test >= n:
            n_val = max(1, n_val - 1)

        val_idx.extend(indices[:n_val])
        test_idx.extend(indices[n_val:n_val + n_test])
        train_idx.extend(indices[n_val + n_test:])

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return train_idx, val_idx, test_idx


class ImageFolderSubset(Dataset):
    def __init__(self, base_dataset: ImageFolder, indices: Sequence[int], transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        path, target = self.base_dataset.samples[real_idx]
        img = self.base_dataset.loader(path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, target

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


## Data loader Imagens

In [16]:
# Célula 2 — DataLoaders para o modelo que usa a imagem

def build_image_dataloaders(
    root_dir: str | Path = "photos/dataset",
    batch_size: int = 32,
    val_split: float = 0.1,
    test_split: float = 0.1,
    seed: int = 42,
    num_workers: int = 4,
):
    root_dir = Path(root_dir)
    base_dataset = ImageFolder(root=str(root_dir))

    train_idx, val_idx, test_idx = stratified_split_indices_3way(
        targets=base_dataset.targets,
        val_split=val_split,
        test_split=test_split,
        seed=seed,
    )

    # Sem augmentation; apenas o preprocessing que você definiu.
    train_transform = transforms.Compose([
        CenterCropSquare(),
        transforms.Resize((500, 500), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
    ])

    eval_transform = transforms.Compose([
        CenterCropSquare(),
        transforms.Resize((500, 500), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),
    ])

    train_dataset = ImageFolderSubset(base_dataset, train_idx, transform=train_transform)
    val_dataset   = ImageFolderSubset(base_dataset, val_idx, transform=eval_transform)
    test_dataset  = ImageFolderSubset(base_dataset, test_idx, transform=eval_transform)

    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    return train_loader, val_loader, test_loader, base_dataset.classes, base_dataset.class_to_idx

## Data load Histogramas

In [17]:
# Célula 3 — histogramas RGB e DataLoaders para o modelo de histograma

def rgb_histogram_tensor(
    img: Image.Image,
    bins: int = 256,
    normalize_hist: bool = False,
) -> torch.Tensor:
    """
    Retorna um vetor 1D com 3*bins valores: [R_hist | G_hist | B_hist].
    Se normalize_hist=True, cada canal soma 1.
    """
    img = img.convert("RGB")
    arr = np.asarray(img)  # shape: H x W x 3

    feats = []
    for c in range(3):
        hist, _ = np.histogram(arr[..., c], bins=bins, range=(0, 256))
        hist = hist.astype(np.float32)
        if normalize_hist:
            hist /= (hist.sum() + 1e-8)
        feats.append(hist)

    return torch.from_numpy(np.concatenate(feats, axis=0))


class HistogramFolderSubset(Dataset):
    def __init__(
        self,
        base_dataset: ImageFolder,
        indices: Sequence[int],
        bins: int = 256,
        normalize_hist: bool = False,
    ):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.bins = bins
        self.normalize_hist = normalize_hist

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        path, target = self.base_dataset.samples[real_idx]
        img = self.base_dataset.loader(path).convert("RGB")
        hist = rgb_histogram_tensor(img, bins=self.bins, normalize_hist=self.normalize_hist)
        return hist, target


def build_histogram_dataloaders(
    root_dir: str | Path = "photos/dataset",
    batch_size: int = 32,
    val_split: float = 0.1,
    test_split: float = 0.1,
    seed: int = 42,
    num_workers: int = 4,
    bins: int = 256,
    normalize_hist: bool = True,
):
    root_dir = Path(root_dir)
    base_dataset = ImageFolder(root=str(root_dir))

    train_idx, val_idx, test_idx = stratified_split_indices_3way(
        targets=base_dataset.targets,
        val_split=val_split,
        test_split=test_split,
        seed=seed,
    )

    train_dataset = HistogramFolderSubset(
        base_dataset, train_idx, bins=bins, normalize_hist=normalize_hist
    )
    val_dataset = HistogramFolderSubset(
        base_dataset, val_idx, bins=bins, normalize_hist=normalize_hist
    )
    test_dataset = HistogramFolderSubset(
        base_dataset, test_idx, bins=bins, normalize_hist=normalize_hist
    )

    pin_memory = torch.cuda.is_available()

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    return train_loader, val_loader, test_loader, base_dataset.classes, base_dataset.class_to_idx

## Imports e hyperparameters para o treinamento dos modelos


In [18]:
# Célula 1 — imports e hiperparâmetros

from __future__ import annotations

import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Ajuste aqui os hiperparâmetros do experimento
HYPERPARAMS = {
    "root_dir": "photos/dataset",
    "batch_size": 32,
    "val_split": 0.10,
    "test_split": 0.10,
    "seed": 42,
    "num_workers": 4,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 1,
    "log_dir": "logs",
    "experiment_name": "image_classifier",
}

## Modelo simples para classificação de imagens (CNN)

In [19]:
# Célula 2 — modelo simples

class SimpleImageClassifier(pl.LightningModule):
    def __init__(self, num_classes: int, hparams: dict):
        super().__init__()
        self.save_hyperparameters(hparams)

        self.features = nn.Sequential(
            nn.BatchNorm2d(3),
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 500 -> 250


            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 250 -> 125


            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

    def _shared_step(self, batch, stage: str):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log(f"{stage}_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )
        return optimizer

## Treinamento do modelo de classificação de imagens

In [20]:
# Célula 3 — dataloaders, logger, trainer e treino

# Usa a função que você já criou antes:
train_loader, val_loader, test_loader, classes, class_to_idx = build_image_dataloaders(
    root_dir=HYPERPARAMS["root_dir"],
    batch_size=HYPERPARAMS["batch_size"],
    val_split=HYPERPARAMS["val_split"],
    test_split=HYPERPARAMS["test_split"],
    seed=HYPERPARAMS["seed"],
    num_workers=HYPERPARAMS["num_workers"],
)
print(f"Classes: {len(classes)}")
model_image = SimpleImageClassifier(
    num_classes=len(classes),
    hparams=HYPERPARAMS,
)

logger = TensorBoardLogger(
    save_dir=HYPERPARAMS["log_dir"],
    name=HYPERPARAMS["experiment_name"],
)

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch:02d}-{val_loss:.4f}",
)

early_stop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
)

trainer = pl.Trainer(
    max_epochs=HYPERPARAMS["max_epochs"],
    logger=logger,
    callbacks=[checkpoint_cb, early_stop_cb],
    accelerator="auto",
    devices="auto",
    log_every_n_steps=10,
    precision="16-mixed"
)

trainer.fit(model_image, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(model_image, dataloaders=test_loader)

print("Classes:", classes)
print("Class to idx:", class_to_idx)
print("TensorBoard logs em:", logger.log_dir)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/bruno/miniconda3/envs/EA979/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name       | Type       | Params | Mode  | FLOPs
----------------------------------------------------------
0 | features   | Sequential | 93.4 K | train | 0    
1 | classifier | Sequential | 11.0 K | train | 0    
----------------------------------------------------------
104 K     Trainable params
0         Non-trainable params
104 K     

Classes: 42
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/bruno/miniconda3/envs/EA979/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 0: 100%|██████████| 193/193 [00:46<00:00,  4.11it/s, v_num=8]        
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 193/193 [00:49<00:00,  3.93it/s, v_num=8, val_loss=1.930, val_acc=0.377, train_loss=2.550, train_acc=0.252]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 193/193 [00:49<00:00,  3.93it/s, v_num=8, val_loss=1.930, val_acc=0.377, train_loss=2.550, train_acc=0.252]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Testing DataLoader 0: 100%|██████████| 24/24 [00:01<00:00, 12.94it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.37962964177131653
        test_loss           1.9607912302017212
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Classes: ['175807_futuro', '175807_maria_e_sobel', '175807_pixelular', '186629_canny_edge_detection', '186629_chromatic_aberration_blur', '186629_color_splash', '237310_aberracao_cromatica', '237310_pixelizacao', '237310_quantizacao', '241163_chromatic_aberration', '241163_edge_detection', '241163_pixelation', '243360_chromatic_aberration', '243360_edge_detection', '243360_radial_blur', '245609_borda_lapis', '245609_fisheye

In [21]:
# Célula 4 — abrir TensorBoard no notebook, se quiser

%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 10313), started 0:08:23 ago. (Use '!kill 10313' to kill it.)

## Histograma modelo

In [26]:


HYPERPARAMS_HIST = {
    "root_dir": "photos/dataset",
    "batch_size": 64,
    "val_split": 0.10,
    "test_split": 0.10,
    "seed": 42,
    "num_workers": 4,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 10,
    "log_dir": "logs",
    "experiment_name": "histogram_classifier",

    # específicos do histograma
    "bins": 128,              # recomendo testar 32 / 64 / 128
    "normalize_hist": True,  # geralmente melhor
    "hidden_dim": 128,
}

## Modelo simples para classificação de histogramas RGB

In [27]:
# Célula 2 — modelo (MLP simples)




class HistogramClassifier(pl.LightningModule):
    def __init__(self, num_classes: int, hparams: dict):
        super().__init__()
        self.save_hyperparameters(hparams)

        input_dim = 3 * self.hparams["bins"]

        self.model = nn.Sequential(
            nn.Linear(input_dim, self.hparams["hidden_dim"]),
            nn.ReLU(),
            nn.Linear(self.hparams["hidden_dim"], self.hparams["hidden_dim"]),
            nn.ReLU(),
            nn.Linear(self.hparams["hidden_dim"], num_classes),
        )

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage: str):
        x, y = batch  # x shape: [B, 3*bins]
        logits = self(x)
        loss = F.cross_entropy(logits, y)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()

        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True)
        self.log(f"{stage}_acc", acc, prog_bar=True, on_epoch=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )

# Célula 3 — dataloaders, logger, trainer e treino do modelo de histograma

In [28]:
# Célula 3 — dataloaders de histograma + treino



train_loader, val_loader, test_loader, classes, class_to_idx = build_histogram_dataloaders(
    root_dir=HYPERPARAMS_HIST["root_dir"],
    batch_size=HYPERPARAMS_HIST["batch_size"],
    val_split=HYPERPARAMS_HIST["val_split"],
    test_split=HYPERPARAMS_HIST["test_split"],
    seed=HYPERPARAMS_HIST["seed"],
    num_workers=HYPERPARAMS_HIST["num_workers"],
    bins=HYPERPARAMS_HIST["bins"],
    normalize_hist=HYPERPARAMS_HIST["normalize_hist"],
)

model_hist = HistogramClassifier(
    num_classes=len(classes),
    hparams=HYPERPARAMS_HIST,
)

logger = TensorBoardLogger(
    save_dir=HYPERPARAMS_HIST["log_dir"],
    name=HYPERPARAMS_HIST["experiment_name"],
)

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch:02d}-{val_loss:.4f}",
)

early_stop_cb = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
)

trainer = pl.Trainer(
    max_epochs=HYPERPARAMS_HIST["max_epochs"],
    logger=logger,
    callbacks=[checkpoint_cb, early_stop_cb],
    accelerator="auto",
    devices="auto",
    log_every_n_steps=10,
)

trainer.fit(model_hist, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(model_hist, dataloaders=test_loader)

print("TensorBoard logs em:", logger.log_dir)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type       | Params | Mode  | FLOPs
-----------------------------------------------------
0 | model | Sequential | 71.2 K | train | 0    
-----------------------------------------------------
71.2 K    Trainable params
0         Non-trainable params
71.2 K    Total params
0.285     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 0: 100%|██████████| 97/97 [00:09<00:00,  9.93it/s, v_num=0, train_loss_step=2.870, train_acc_step=0.233] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 97/97 [00:08<00:00, 11.56it/s, v_num=0, train_loss_step=2.370, train_acc_step=0.300, val_loss=2.820, val_acc=0.257, train_loss_epoch=3.420, train_acc_epoch=0.0912]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 97/97 [00:08<00:00, 11.37it/s, v_num=0, train_loss_step=1.960, train_acc_step=0.500, val_loss=2.190, val_acc=0.392, train_loss_epoch=2.460, train_acc_epoch=0.282] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 97/97 [00:08<00:00, 11.38it/s, v_num=0, train_loss_step=1.980, train_acc_step=0.367, val_loss=1.980, val_acc=0.362, train_loss_epoch=2.070, train_acc_epoch=0.368]
Validation: |          | 0/? [00:00<?, ?it/

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 97/97 [00:09<00:00,  9.81it/s, v_num=0, train_loss_step=1.480, train_acc_step=0.600, val_loss=1.560, val_acc=0.529, train_loss_epoch=1.560, train_acc_epoch=0.517]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Testing DataLoader 0: 100%|██████████| 12/12 [00:00<00:00, 14.90it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.5105820298194885
        test_loss           1.5682778358459473
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
TensorBoard logs em: logs/histogram_classifier/version_0


In [30]:
%load_ext tensorboard
%tensorboard --logdir logs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 10313), started 0:24:38 ago. (Use '!kill 10313' to kill it.)

## Ensemble simples combinando os dois modelos


In [29]:
class CombinedDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        indices,
        image_transform,
        bins=64,
        normalize_hist=True,
    ):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.image_transform = image_transform
        self.bins = bins
        self.normalize_hist = normalize_hist

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        path, target = self.base_dataset.samples[real_idx]

        img = self.base_dataset.loader(path).convert("RGB")

        # imagem (CNN)
        x_img = self.image_transform(img)

        # histograma
        x_hist = rgb_histogram_tensor(
            img,
            bins=self.bins,
            normalize_hist=self.normalize_hist,
        )

        return x_img, x_hist, target